In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgbm

In [7]:
# 데이터 분할 및 전처리
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing        import OneHotEncoder
from sklearn.compose              import ColumnTransformer

# 특성 선택
from sklearn.feature_selection import SequentialFeatureSelector

# 하이퍼파라미터 튜닝
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import precision_recall_curve
# 모델 평가 지표
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

# 데이터 시각화
import seaborn as sns

### 1. 데이터 로드 및 탐색

#### 1.1 데이터 로드

In [8]:
df_trn = pd.read_csv('train.csv')
df_tst = pd.read_csv('test.csv')

### 2. 데이터 전처리

#### 2.1 범주형 데이터 인코딩

##### Unknown 처리

In [6]:
cat_cols = ['job','marital','contact','poutcome','education']

# train 에서 비율 5% 미만 + 'unknown'은 모두 'other'로 묶기
for col in cat_cols:
    freq = df_trn[col].value_counts(normalize=True)
    rare = freq[freq < 0.05].index.tolist()
    if 'unknown' in df_trn[col].unique():
        rare.append('unknown')
    # train/test 모두 반영
    df_trn[col] = df_trn[col].replace(rare, 'other')
    df_tst[col] = df_tst[col].replace(rare, 'other')

# 나머지 범주형(yes/no형)도 포함해서 OHE
ohe_cols = cat_cols + ['default','housing','loan','month']
ohe = OneHotEncoder(drop='first', sparse=False, handle_unknown='ignore')
preproc = ColumnTransformer(
    transformers=[
        ('ohe', ohe, ohe_cols),
    ],
    remainder='passthrough'   # 나머지 컬럼은 그대로 둠
)

NameError: name 'OneHotEncoder' is not defined

In [9]:
# 범주형 데이터를 라벨 인코딩
cat = ['job','marital','contact','poutcome','education','default','housing','loan','month']
label_encoder = {}
for col in cat:
    le = LabelEncoder()
    df_trn[col] = le.fit_transform(df_trn[col])
    df_tst[col] = le.transform(df_tst[col])
    label_encoder[col] = le

#### 2.2 파생 변수 생성

In [10]:
# 계절을 반환하는 함수 정의
def get_season(month):
    if 3 <= month <= 5:
        return 'spring'
    elif 6 <= month <= 8:
        return 'summer'
    elif 9 <= month <= 11:
        return 'autumn'
    else:
        return 'winter'

In [15]:
# 데이터 전처리 함수 정의
def preprocess_data(df_trn, df_tst):

    le = LabelEncoder()
    
    for df in [df_trn, df_tst]:

        # pdays 를 갖고 연락여부로 이진분류한 새로운 칼럼 생성
        df['pdays_contacted'] = (df['pdays'] != -1).astype(int)
        
        # 흠....
        df['duration_per_campaign'] = df['duration'] / (df['campaign'] + 1)

        # day와 month를 통합하여 연속적인 시간 정보를 제공하기 위한 새로운 칼럼 생성
        df['day_of_year'] = df['day'] + (df['month'] - 1) * 30
        
        # 계별 특성을 학습할 수 있도록 새로운 칼럼 생성
        df['season'] = df['month'].apply(get_season)
        
        #month 변수에 주기성 추가
        month_map = {'jan':1,'feb':2,'mar':3,'apr':4,'may':5,'jun':6,
                    'jul':7,'aug':8,'sep':9,'oct':10,'nov':11,'dec':12}
        df['month_num'] = df['month'].map(month_map)
        df['month_sin'] = np.sin((df['month_num']-1)/12 * 2*np.pi)
        df['month_cos'] = np.cos((df['month_num']-1)/12 * 2*np.pi)

        #balance 로그 변환, 캠페인 비율
        df['balance_log']     = np.log1p(df['balance'].clip(lower=0))
        df['campaign_ratio']  = df['campaign'] / (df['previous'] + 1)

        # 대출 통합 housing + loan
        df['has_any_loan'] = ((df['housing']=='yes') | (df['loan']=='yes')).astype(int)

        # job + education 조합
        df['job_edu'] = df['job'].astype(str) + '_' + df['education'].astype(str)

        # pdays 구간화
        conds = [
            df['pdays']==-1,
            df['pdays'].between(0,90),
            df['pdays'].between(91,365),
            df['pdays']>365
        ]
        choices = ['none','0-90','91-365','>365']
        df['pdays_cat'] = np.select(conds, choices, default='unknown')
        
        #인코딩 필요한 칼럼들 정의
        for col in ['season','pdays_cat','job_edu']:
            df[col] = le.fit_transform(df[col].astype(str))
    
    return df_trn, df_tst

In [17]:
# 전처리 함수 적용 후 할당
df_trn_preprocessed, df_tst_preprocessed = preprocess_data(df_trn,df_tst)

#### 2.3 Feature와 Target 분리

In [18]:
total_cols = [x for x in df_trn.columns]

id_cols = ['ID']
y_cols = ['label']
x_cols = [x for x in total_cols if x not in id_cols+y_cols]

In [21]:
# feature와 target분리
df_trn_x = df_trn_preprocessed[x_cols]
df_trn_y = df_trn_preprocessed[y_cols].values.ravel()
df_tst_x = df_tst_preprocessed[x_cols]

In [22]:
df_trn_x.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31647 entries, 0 to 31646
Data columns (total 28 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   age                    31647 non-null  int64  
 1   job                    31647 non-null  int64  
 2   marital                31647 non-null  int64  
 3   education              31647 non-null  int64  
 4   default                31647 non-null  int64  
 5   balance                31647 non-null  int64  
 6   housing                31647 non-null  int64  
 7   loan                   31647 non-null  int64  
 8   contact                31647 non-null  int64  
 9   day                    31647 non-null  int64  
 10  month                  31647 non-null  int64  
 11  duration               31647 non-null  int64  
 12  campaign               31647 non-null  int64  
 13  pdays                  31647 non-null  int64  
 14  previous               31647 non-null  int64  
 15  po

### 3. 모델 정의 및 학습

#### 3.1 검증 데이터 분리

In [23]:
# 검증 데이터셋 분리
df_retrn_x, df_val_x, df_retrn_y, df_val_y = train_test_split(df_trn_x, df_trn_y, test_size=0.2, random_state=42, stratify=df_trn_y)

#### 3.2 초기 모델 정의 및 학습

In [24]:
# 초기 모델 정의
base_model = lgbm.LGBMClassifier(random_state=42, verbose=-1, class_weight='balanced')
#학습
base_model.fit(df_retrn_x, df_retrn_y)

LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1)

#### 3.3 변수 중요도 분석

In [25]:
# Feature Importance 추출
importances = base_model.feature_importances_
features = df_retrn_x.columns

# 중요도를 데이터프레임으로 저장
importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# 출력
print(importance_df)

                  Feature  Importance
18            day_of_year         606
11               duration         407
17  duration_per_campaign         247
0                     age         240
5                 balance         219
9                     day         200
13                  pdays         163
10                  month         113
26                job_edu         107
8                 contact          86
23            balance_log          73
15               poutcome          70
1                     job          67
6                 housing          64
3               education          61
24         campaign_ratio          56
19                 season          49
12               campaign          44
2                 marital          43
14               previous          40
7                    loan          33
4                 default           6
27              pdays_cat           6
20              month_num           0
21              month_sin           0
22          

In [34]:
from sklearn.preprocessing import OneHotEncoder

from sklearn.compose    import ColumnTransformer


# 1) 불필요 컬럼 드롭
drop_cols = ['month','month_num','pdays','housing','loan']
X_trn = df_retrn_x.drop(columns=drop_cols)
X_val = df_val_x.drop(columns=drop_cols)

# 2) 카테고리형 파생 변수만 One-Hot
cat_derived = ['job_edu','pdays_cat','season']
num_derived = ['month_sin','month_cos','balance_log','campaign_ratio','duration_per_campaign','day_of_year']

preproc = ColumnTransformer([
    ('ohe', OneHotEncoder(drop='first', sparse_output=False), cat_derived),
], remainder='passthrough')

X_trn_enc = preproc.fit_transform(X_trn, df_retrn_y)
X_val_enc = preproc.transform(X_val)

# 컬럼 이름 복원 (나중에 중요도 확인 편하게)
ohe_cols = preproc.named_transformers_['ohe'].get_feature_names_out(cat_derived)
all_cols = list(ohe_cols) + [c for c in X_trn.columns if c not in cat_derived]

X_trn_enc = pd.DataFrame(X_trn_enc, columns=all_cols)
X_val_enc = pd.DataFrame(X_val_enc, columns=all_cols)

# 3) 모델 학습 & 중요도 확인
model = lgbm.LGBMClassifier(**best_params, random_state=42, class_weight='balanced')
model.fit(X_trn_enc, df_retrn_y)

importances = pd.Series(model.feature_importances_, index=all_cols)
imp_derived = importances.loc[num_derived + list(ohe_cols)].sort_values(ascending=False)

imp_derived.plot.bar(figsize=(6,3), title='Refined Derived Importances')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

NameError: name 'best_params' is not defined

#### 3.4 변수 선택법

In [16]:
# 변수 선택 모델 정의
sfs = SequentialFeatureSelector(
    base_model,
    n_features_to_select='auto',
    direction='forward',
    scoring='f1',
    cv=5,
    n_jobs=-1
)
# 변수 선택 모델 학습하기
sfs.fit(df_retrn_x, df_retrn_y)

/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted w

KeyboardInterrupt: 

In [17]:
# 선택된 변수 할당하기
#selected_features = df_retrn_x.columns[sfs.get_support()].tolist()
selected_features = ['age', 'housing', 'contact', 'day', 'duration', 'campaign', 'pdays', 'poutcome', 'duration_per_campaign', 'day_of_year']
print(f"선택된 변수: {selected_features}")

선택된 변수: ['age', 'housing', 'contact', 'day', 'duration', 'campaign', 'pdays', 'poutcome', 'duration_per_campaign', 'day_of_year']


In [18]:
# 선택된 변수로 데이터 재구성
df_retrn_x_selected = df_retrn_x[selected_features]
df_val_x_selected = df_val_x[selected_features]
df_tst_x_selected = df_tst_x[selected_features]

### 4. 모델 최적화

#### 4.1 OPTUNA

In [19]:
X_train, X_val = df_retrn_x_selected, df_val_x_selected
y_train, y_val = df_retrn_y, df_val_y

In [21]:
import optuna
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score
from lightgbm.callback import early_stopping, log_evaluation


In [22]:
def objective(trial):
#탐색할 하이퍼파라미터 범위 설정
    param = {
        'class_weight': 'balanced',
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 1e-1),
        'num_leaves': trial.suggest_int('num_leaves', 16, 128),        
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 200),
        'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-3, 10.0),
        'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-3, 10.0),
        'random_state': 42,
        'verbose': -1,
    }

    # 모델 학습 (early stopping 활용)
    model = LGBMClassifier(**param)
    model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='f1',
    callbacks=[
        early_stopping(stopping_rounds=100),   
        log_evaluation(period=0)            
    ]
)

    # 3.3 최종 평가
    preds = model.predict(X_val)
    return f1_score(y_val, preds)


In [26]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, timeout=600)  


[I 2025-05-27 20:30:41,814] A new study created in memory with name: no-name-9578da21-c008-419b-9c72-4706ecc05aee


Training until validation scores don't improve for 100 rounds


[I 2025-05-27 20:30:42,851] Trial 0 finished with value: 0.6073558648111332 and parameters: {'n_estimators': 444, 'learning_rate': 0.25704440759673175, 'max_depth': 5, 'num_leaves': 116, 'min_child_samples': 182, 'subsample': 0.9647458043864123, 'colsample_bytree': 0.7405186090610825, 'reg_alpha': 0.0012212402149659326, 'reg_lambda': 2.70535397527658}. Best is trial 0 with value: 0.6073558648111332.


Did not meet early stopping. Best iteration is:
[442]	valid_0's binary_logloss: 0.295163
Training until validation scores don't improve for 100 rounds


[I 2025-05-27 20:30:43,096] Trial 1 finished with value: 0.4300341296928328 and parameters: {'n_estimators': 142, 'learning_rate': 0.002015300870172563, 'max_depth': 3, 'num_leaves': 114, 'min_child_samples': 196, 'subsample': 0.903188762034572, 'colsample_bytree': 0.62277542447269, 'reg_alpha': 6.302346760653145, 'reg_lambda': 0.036824133584862084}. Best is trial 0 with value: 0.6073558648111332.


Did not meet early stopping. Best iteration is:
[142]	valid_0's binary_logloss: 0.62185
Training until validation scores don't improve for 100 rounds


[I 2025-05-27 20:30:44,484] Trial 2 finished with value: 0.5963203463203464 and parameters: {'n_estimators': 647, 'learning_rate': 0.5831097588175106, 'max_depth': 12, 'num_leaves': 85, 'min_child_samples': 152, 'subsample': 0.5688267940793317, 'colsample_bytree': 0.9403530361495338, 'reg_alpha': 0.03051040986048851, 'reg_lambda': 2.290799658322645}. Best is trial 0 with value: 0.6073558648111332.


Early stopping, best iteration is:
[74]	valid_0's binary_logloss: 0.291079
Training until validation scores don't improve for 100 rounds


[I 2025-05-27 20:30:48,964] Trial 3 finished with value: 0.5178127589063795 and parameters: {'n_estimators': 986, 'learning_rate': 0.0021337707163253568, 'max_depth': 6, 'num_leaves': 177, 'min_child_samples': 50, 'subsample': 0.6283495086577953, 'colsample_bytree': 0.6974696016327061, 'reg_alpha': 7.2546108897485, 'reg_lambda': 5.781676113425715}. Best is trial 0 with value: 0.6073558648111332.


Did not meet early stopping. Best iteration is:
[986]	valid_0's binary_logloss: 0.407309
Training until validation scores don't improve for 100 rounds


In [24]:
#best_params = study.best_trial.params
best_params = {'n_estimators': 680, 'learning_rate': 0.18802531719522447, 'max_depth': 11, 'num_leaves': 77, 'min_child_samples': 97, 'subsample': 0.9125189892140894, 'colsample_bytree': 0.9036442293342724, 'reg_alpha': 1.7613349573900532, 'reg_lambda': 0.3367791707298721}
print("Best params:", best_params)


Best params: {'n_estimators': 680, 'learning_rate': 0.18802531719522447, 'max_depth': 11, 'num_leaves': 77, 'min_child_samples': 97, 'subsample': 0.9125189892140894, 'colsample_bytree': 0.9036442293342724, 'reg_alpha': 1.7613349573900532, 'reg_lambda': 0.3367791707298721}


In [31]:
# Best params: {'n_estimators': 680, 'learning_rate': 0.18802531719522447, 'max_depth': 11, 'num_leaves': 77, 'min_child_samples': 97, 'subsample': 0.9125189892140894, 'colsample_bytree': 0.9036442293342724, 'reg_alpha': 1.7613349573900532, 'reg_lambda': 0.3367791707298721}
# optuna를 진행해보니 val_f1 score : 0.5569

#### optuna 시각화

In [27]:
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate,
    plot_slice,
    plot_contour
)

In [28]:
# 1) Trial별 F1-score 추이
fig1 = plot_optimization_history(study)
fig1.show()

# 2) Hyperparameter 중요도 분석
fig2 = plot_param_importances(study)
fig2.show()

# 3) 두 파라미터 조합별 성능 분포 (parallel coordinate)
fig3 = plot_parallel_coordinate(study, params=['learning_rate','n_estimators','max_depth'])
fig3.show()

# 4) 특정 파라미터 주위 성능 (slice plot)
fig4 = plot_slice(study, params=['learning_rate','subsample'])
fig4.show()

# 5) 2차원 성능 등고선 (contour plot)
fig5 = plot_contour(study, params=['learning_rate','num_leaves'])
fig5.show()

#### optuna 그래프 해석

* 파란 점(Objective Value): 각 Trial(실험)마다 얻은 Validation F1-score

* 빨간 선(Best Value): 지금까지 최고 성능이 유지된 값을 이어 그린 것

+ Trial 1→2에서 급격히 성능이 올라간 뒤, 대략 0.60∼0.62 사이에서 **플래토(plateau)** 에 진입

+ 이후 탐색 공간 전반에서 비슷한 성능이 반복 → 지금 범위 내에서는 큰 도약이 어려움을 의미

**learning_rate (≈0.45) 가 가장 큰 영향력을 보임**

* 그다음으로 n_estimators (≈0.19), min_child_samples (≈0.16)

* num_leaves, max_depth 중간,

* subsample, colsample_bytree, reg_alpha, reg_lambda 순으로 중요도 급감

* 색이 진할수록(F1 높을수록), 해당 Trial의 파라미터 값이 어떻게 분포하는지 선으로 이어 줌

* learning_rate ≈ 0.1 ∼ 0.2, max_depth ≈ 5 ∼ 8, n_estimators ≈ 600 ∼ 900 구간에서 짙은 선들이 몰려 있는 모습이 보임



* 왼쪽: learning_rate vs Objective

    * 0.01 미만 구간은 비교적 성능 저조

    * 0.05 ∼ 0.2 구간에서 F1이 전반적으로 높음

* 오른쪽: subsample vs Objective

    * 0.6 ∼ 0.8 사이에서 좋은 Trial이 많고,

    * 0.9 이상에서는 약간 성능이 떨어지는 경향

* learning_rate (x축) 와 num_leaves (y축) 의 조합별 성능 등고선

    * 중간 빛(≈0.60) 영역보다 진한 파란색(≈0.62) 영역이

* learning_rate ≈ 0.1–0.2 × num_leaves ≈ 80–150 구역에 몰려 있음

#### 4.2 최적의 threshold 찾기

In [25]:
#최적의 params 모델에 적용
best_model = LGBMClassifier(**best_params, random_state=42)

#train 데이터 로 학습
best_model.fit(df_retrn_x_selected, df_retrn_y)

# 예측 확률값
y_prob = best_model.predict_proba(df_val_x_selected)[:, 1]  # 양성 클래스 확률

# Precision, Recall, Threshold 계산
precisions, recalls, thresholds = precision_recall_curve(df_val_y, y_prob)

# F1 Score 계산
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)

# 최적 Threshold 선택
best_threshold = thresholds[np.argmax(f1_scores)]
print(f"Best Threshold for F1 Score: {best_threshold}")

# 최적 Threshold 기반 예측
y_pred = (y_prob >= best_threshold).astype(int)

# F1 Score 확인
print(f"Optimized F1 Score: {f1_score(df_val_y, y_pred)}")


Best Threshold for F1 Score: 0.1855294046582513
Optimized F1 Score: 0.6052060737527115


In [ ]:
#Best Threshold for F1 Score: 0.1855294046582513
#Optimized F1 Score: 0.6052060737527115

### 5. 모델 평가

In [29]:
# threshold 설정
thres=0.1855

In [30]:
# 훈련 데이터에 대한 성능 평가
y_retrn_pred_proba = best_model.predict_proba(df_retrn_x_selected)[:, 1]
y_retrn_pred_label = (y_retrn_pred_proba > thres).astype(int)

print(f"훈련 데이터 Accuracy: {accuracy_score(df_retrn_y, y_retrn_pred_label)}")
print(f"훈련 데이터 Precision: {precision_score(df_retrn_y, y_retrn_pred_label)}")
print(f"훈련 데이터 Recall: {recall_score(df_retrn_y, y_retrn_pred_label)}")
print(f"훈련 데이터 F1 Score: {f1_score(df_retrn_y, y_retrn_pred_label)}")

훈련 데이터 Accuracy: 0.9594343721609986
훈련 데이터 Precision: 0.743151545614476
훈련 데이터 Recall: 0.9983119513841998
훈련 데이터 F1 Score: 0.852038611151131


검증 데이터로 성능 평가

In [31]:
# 검증 데이터에 대한 성능 평가
y_val_pred_proba = best_model.predict_proba(df_val_x_selected)[:, 1]
y_val_pred_label = (y_val_pred_proba > thres).astype(int)

print(f"검증 데이터 Accuracy: {accuracy_score(df_val_y, y_val_pred_label)}")
print(f"검증 데이터 Precision: {precision_score(df_val_y, y_val_pred_label)}")
print(f"검증 데이터 Recall: {recall_score(df_val_y, y_val_pred_label)}")
print(f"검증 데이터 F1 Score: {f1_score(df_val_y, y_val_pred_label)}")

검증 데이터 Accuracy: 0.8848341232227488
검증 데이터 Precision: 0.5049773755656108
검증 데이터 Recall: 0.754054054054054
검증 데이터 F1 Score: 0.6048780487804878


### 6. 최종 모델 학습(전체 훈련 데이터)

In [1]:
df_trn_x_selected = df_trn_x[selected_features]
best_model.fit(df_trn_x_selected, df_trn_y)

NameError: name 'df_trn_x' is not defined

### 7. 테스트 데이터에 대한 예측 후 저장하기

#### 7.1 테스트데이터에 대한 예측 수행

In [ ]:
y_tst_pred_proba = best_model.predict_proba(df_tst_x_selected)[:,1]

In [ ]:
y_tst_pred_label = (y_tst_pred_proba > thres).astype(int)

#### 7.2 예측결과 저장하기

In [ ]:
df_submission = pd.read_csv('Ho_Optuna_try01')

In [ ]:
df_submission.head()

,ID,label
0,test00001,0
1,test00002,0
2,test00003,0
3,test00004,0
4,test00005,0


In [ ]:
df_submission['label'] = y_tst_pred_label

In [ ]:
df_submission['label'].value_counts()

label
0    11541
1     2023
Name: count, dtype: int64

In [ ]:
# 다하고 눌러라!!!!
filename = 'LightGBM_Ho_final-1.csv'
df_submission.to_csv(filename, index=False)